In [6]:
import json

import pandas as pd

TEAM_NAME = 'Napoli'
MIN_ATTEMPTS = 30

matches = pd.read_csv('data/matches.csv')
allmatches = matches[(matches['home'] == TEAM_NAME) | (matches['away'] == TEAM_NAME)]


In [7]:
match_rows = []
player_rows = []

for _, match_row in allmatches.iterrows():
    match_id = match_row['statsbomb']
    opponent = match_row['away'] if match_row['home'] == TEAM_NAME else match_row['home']

    with open(f"data/statsbomb/league_phase/{match_id}.json", 'r', encoding='utf-8') as f:
        events = json.load(f)

    df = pd.json_normalize(events)
    match_passes = df[(df['type.id'] == 30) & (df['team.name'] == TEAM_NAME)].copy()
    match_passes['successful'] = match_passes['pass.outcome.name'].isna()

    passes_attempted = len(match_passes)
    passes_completed = int(match_passes['successful'].sum())

    player_summary = match_passes.groupby('player.name').size().reset_index(name='attempted')
    player_completed = match_passes[match_passes['successful']].groupby('player.name').size().reset_index(name='completed')
    player_summary = player_summary.merge(player_completed, on='player.name', how='left')
    player_summary['completed'] = player_summary['completed'].fillna(0).astype(int)
    player_summary = player_summary.rename(columns={'player.name': 'player'})
    player_summary['success_pct'] = player_summary['completed'] / player_summary['attempted'] * 100
    player_summary['team'] = TEAM_NAME
    player_summary['match_id'] = match_id
    player_summary['opponent'] = opponent
    player_rows.append(player_summary)

    match_rows.append({
        'match_id': match_id,
        'opponent': opponent,
        'passes_attempted': passes_attempted,
        'passes_completed': passes_completed,
        'success_rate_pct': passes_completed / passes_attempted * 100
    })

match_table = pd.DataFrame(match_rows).sort_values('match_id').reset_index(drop=True)
all_player_matches = pd.concat(player_rows, ignore_index=True)
all_player_matches = all_player_matches[all_player_matches['team'] == TEAM_NAME]

player_table = all_player_matches.groupby('player', as_index=False)[['attempted', 'completed']].sum()
player_table['success_pct'] = player_table['completed'] / player_table['attempted'] * 100
player_table = player_table.sort_values(['success_pct', 'attempted'], ascending=[False, False]).reset_index(drop=True)

eligible_players = player_table[player_table['attempted'] >= MIN_ATTEMPTS]
top_player = eligible_players.iloc[0]

print(f"Top player by successful pass % (min {MIN_ATTEMPTS} attempts): {top_player['player']} - {top_player['success_pct']:.2f}%")

print('\nMatch-level output table:')
display(match_table)

print('\nPlayer-level output table:')
display(player_table)


Top player by successful pass % (min 30 attempts): Stanislav Lobotka - 95.51%

Match-level output table:


,match_id,opponent,passes_attempted,passes_completed,success_rate_pct
0,4028820,Qarabag,576,501,86.979167
1,4028848,Manchester City,310,242,78.064516
2,4028897,Sporting CP,553,489,88.426763
3,4028900,PSV,388,316,81.443299
4,4028923,Frankfurt,736,643,87.364130
5,4028951,Benfica,570,464,81.403509
6,4028957,København,757,664,87.714663
7,4028970,Chelsea,570,487,85.438596



Player-level output table:


,player,attempted,completed,success_pct
0,Stanislav Lobotka,379,362,95.514512
1,Amir Rrahmani,253,236,93.280632
2,Sam Beukema,263,241,91.634981
3,Alessandro Buongiorno,430,387,90.000000
4,Juan Guilherme Nunes Jesus,273,244,89.377289
5,Scott McTominay,348,306,87.931034
6,Mathías Olivera Miramontes,180,158,87.777778
7,Giovanni Di Lorenzo,442,387,87.556561
8,Billy Gilmour,64,56,87.500000
9,Miguel Gutiérrez Ortega,211,183,86.729858
